# 03 SQL Analysis and Validation
Builds a SQLite database from the cleaned data, runs the analysis queries, and checks that Python, SQL and Power BI agree.

In [1]:
import sqlite3
import pandas as pd

df_clean = pd.read_csv(
    "../data/cleaned/superstore_cleaned.csv",
    parse_dates=["Order Date", "Ship Date"],
    dtype={"Postal Code": str},
)

sql_df = df_clean.copy()

# Column names: lowercase, underscores instead of spaces/hyphens
sql_df.columns = (sql_df.columns.str.lower()
                  .str.replace(" ", "_")
                  .str.replace("-", "_"))

# Dates as plain text like 2016-11-08 (SQLite's date format)
sql_df["order_date"] = sql_df["order_date"].dt.strftime("%Y-%m-%d")
sql_df["ship_date"] = sql_df["ship_date"].dt.strftime("%Y-%m-%d")

conn = sqlite3.connect("../data/superstore.db")
sql_df.to_sql("sales", conn, if_exists="replace", index=False)

print(sql_df.shape)

(9993, 21)


In [2]:
customers = sql_df[["customer_id", "customer_name", "segment"]].drop_duplicates()
print("Rows in customers table:", len(customers))
print("Unique customer IDs:   ", sql_df["customer_id"].nunique())

customers.to_sql("customers", conn, if_exists="replace", index=False)

def q(sql):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql_query(sql, conn)

q("SELECT COUNT(*) AS total_rows FROM sales")

Rows in customers table: 793
Unique customer IDs:    793


,total_rows
0,9993


In [3]:
import sqlite3

# Copy of the cleaned data prepared for SQL
sql_df = df_clean.copy()

# 1) Column names: lowercase, with underscores instead of spaces/hyphens
sql_df.columns = (sql_df.columns.str.lower()
                  .str.replace(" ", "_")
                  .str.replace("-", "_"))

# 2) Drop helper columns we added during EDA (they are not original data)
sql_df = sql_df.drop(columns=["year", "yearmonth"], errors="ignore")

# 3) Store dates as plain text like 2016-11-08 (SQLite's date format)
sql_df["order_date"] = sql_df["order_date"].dt.strftime("%Y-%m-%d")
sql_df["ship_date"] = sql_df["ship_date"].dt.strftime("%Y-%m-%d")

# 4) Create the database file and write the table
conn = sqlite3.connect("../data/superstore.db")
sql_df.to_sql("sales", conn, if_exists="replace", index=False)

print(sql_df.shape)
print(list(sql_df.columns))

(9993, 21)
['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


In [4]:
# Python results
py = {
    "Total Sales": round(df_clean["Sales"].sum(), 2),
    "Total Profit": round(df_clean["Profit"].sum(), 2),
    "Total Orders": df_clean["Order ID"].nunique(),
    "Profit Margin %": round(df_clean["Profit"].sum() / df_clean["Sales"].sum() * 100, 2),
}

# SQL results
s = q("""
SELECT ROUND(SUM(sales), 2)                     AS a,
       ROUND(SUM(profit), 2)                    AS b,
       COUNT(DISTINCT order_id)                 AS c,
       ROUND(SUM(profit) / SUM(sales) * 100, 2) AS d
FROM sales
""").iloc[0]
sql = {"Total Sales": s["a"], "Total Profit": s["b"],
       "Total Orders": s["c"], "Profit Margin %": s["d"]}

result = pd.DataFrame({"Python": py, "SQL": sql})
result["Match"] = (result["Python"] - result["SQL"]).abs() < 0.01
result

,Python,SQL,Match
Total Sales,2296919.49,2296919.49,True
Total Profit,286409.08,286409.08,True
Total Orders,5009.00,5009.00,True
Profit Margin %,12.47,12.47,True


In [5]:
q("""
SELECT ROUND(SUM(s.sales), 2) AS total_sales, ROUND(SUM(s.profit), 2) AS total_profit
FROM sales AS s
INNER JOIN customers AS c ON s.customer_id = c.customer_id
""")

,total_sales,total_profit
0,2296919.49,286409.08


In [6]:
d17 = df_clean[df_clean["Order Date"].dt.year == 2017]
print("2017 orders:", d17["Order ID"].nunique())
print("West profit:", round(df_clean[df_clean["Region"] == "West"]["Profit"].sum(), 2))

2017 orders: 1687
West profit: 108418.45


In [7]:
q("""
SELECT ROUND(SUM(s.sales), 2) AS total_sales, ROUND(SUM(s.profit), 2) AS total_profit
FROM sales AS s
INNER JOIN customers AS c ON s.customer_id = c.customer_id
""")

,total_sales,total_profit
0,2296919.49,286409.08


In [8]:
q("""
SELECT
    c.segment,
    ROUND(SUM(s.sales), 2)  AS total_sales,
    ROUND(SUM(s.profit), 2) AS total_profit,
    ROUND(SUM(s.profit) / SUM(s.sales) * 100, 2) AS margin_pct
FROM sales AS s
INNER JOIN customers AS c ON s.customer_id = c.customer_id
GROUP BY c.segment
ORDER BY total_profit DESC
""")

,segment,total_sales,total_profit,margin_pct
0,Consumer,1161401.34,134119.21,11.55
1,Corporate,706146.37,91979.13,13.03
2,Home Office,429371.78,60310.74,14.05


In [9]:
seg = df_clean.groupby("Segment")[["Sales", "Profit"]].sum()
seg["Margin %"] = seg["Profit"] / seg["Sales"] * 100
seg.sort_values("Profit", ascending=False).round(2)

,Sales,Profit,Margin %
Segment,,,
Consumer,1161401.34,134119.21,11.55
Corporate,706146.37,91979.13,13.03
Home Office,429371.78,60310.74,14.05


In [10]:
d17 = df_clean[df_clean["Order Date"].dt.year == 2017]
print("2017 orders:", d17["Order ID"].nunique())
print("West profit:", round(df_clean[df_clean["Region"] == "West"]["Profit"].sum(), 2))

2017 orders: 1687
West profit: 108418.45


In [11]:
customers = sql_df[["customer_id", "customer_name", "segment"]].drop_duplicates()

print("Rows in customers table:", len(customers))
print("Unique customer IDs:   ", sql_df["customer_id"].nunique())

Rows in customers table: 793
Unique customer IDs:    793


In [12]:
customers.to_sql("customers", conn, if_exists="replace", index=False)

793

In [13]:
def q(sql):
    """Run a SQL query and return the result as a table."""
    return pd.read_sql_query(sql, conn)

q("SELECT COUNT(*) AS total_rows FROM sales")

,total_rows
0,9993


In [14]:
q("""
SELECT
    ROUND(SUM(sales), 2)                  AS total_sales,
    ROUND(SUM(profit), 2)                 AS total_profit,
    COUNT(DISTINCT order_id)              AS total_orders,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) AS profit_margin_pct
FROM sales
""")

,total_sales,total_profit,total_orders,profit_margin_pct
0,2296919.49,286409.08,5009,12.47


In [15]:
q("""
SELECT order_id, order_date, product_name, sales, discount, profit
FROM sales
WHERE profit < 0 AND discount >= 0.5
ORDER BY profit
LIMIT 10
""")

,order_id,order_date,product_name,sales,discount,profit
0,CA-2016-108196,2016-11-25,Cubify CubeX 3D Printer Double Head Print,4499.985,0.7,-6599.9780
1,US-2017-168116,2017-11-04,Cubify CubeX 3D Printer Triple Head Print,7999.980,0.5,-3839.9904
2,CA-2014-169019,2014-07-26,GBC DocuBind P400 Electric Binding System,2177.584,0.8,-3701.8928
3,CA-2017-134845,2017-04-17,Lexmark MX611dhe Monochrome Laser Printer,2549.985,0.7,-3399.9800
4,US-2017-122714,2017-12-07,Ibico EPK-21 Electric Binding System,1889.990,0.8,-2929.4845
5,CA-2015-147830,2015-12-15,Cubify CubeX 3D Printer Double Head Print,1799.994,0.7,-2639.9912
6,CA-2017-131254,2017-11-19,Fellowes PB500 Electric Punch Plastic Comb Bin...,1525.188,0.8,-2287.7820
7,CA-2016-130946,2016-04-08,GBC DocuBind P400 Electric Binding System,1088.792,0.8,-1850.9464
8,CA-2014-145317,2014-03-18,Cisco TelePresence System EX90 Videoconferenci...,22638.480,0.5,-1811.0784
9,US-2015-150630,2015-09-17,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.430,0.5,-1665.0522


In [16]:
df_clean[(df_clean["Profit"] < 0) & (df_clean["Discount"] >= 0.5)].nsmallest(10, "Profit")[["Order ID", "Product Name", "Profit"]]

,Order ID,Product Name,Profit
7771,CA-2016-108196,Cubify CubeX 3D Printer Double Head Print,-6599.9780
683,US-2017-168116,Cubify CubeX 3D Printer Triple Head Print,-3839.9904
9773,CA-2014-169019,GBC DocuBind P400 Electric Binding System,-3701.8928
3011,CA-2017-134845,Lexmark MX611dhe Monochrome Laser Printer,-3399.9800
4990,US-2017-122714,Ibico EPK-21 Electric Binding System,-2929.4845
3151,CA-2015-147830,Cubify CubeX 3D Printer Double Head Print,-2639.9912
5309,CA-2017-131254,Fellowes PB500 Electric Punch Plastic Comb Bin...,-2287.7820
1199,CA-2016-130946,GBC DocuBind P400 Electric Binding System,-1850.9464
2697,CA-2014-145317,Cisco TelePresence System EX90 Videoconferenci...,-1811.0784
27,US-2015-150630,"Riverside Palais Royal Lawyers Bookcase, Royal...",-1665.0522


In [17]:
q("""
SELECT
    category,
    COUNT(*)                     AS rows_count,
    ROUND(SUM(sales), 2)         AS total_sales,
    ROUND(SUM(profit), 2)        AS total_profit,
    ROUND(AVG(discount), 3)      AS avg_discount,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) AS margin_pct
FROM sales
GROUP BY category
ORDER BY total_sales DESC
""")

,category,rows_count,total_sales,total_profit,avg_discount,margin_pct
0,Technology,1847,836154.03,145454.95,0.132,17.40
1,Furniture,2120,741718.42,18463.33,0.174,2.49
2,Office Supplies,6026,719047.03,122490.80,0.157,17.04


In [18]:
q("""
SELECT
    region,
    ROUND(SUM(sales), 2)   AS total_sales,
    ROUND(SUM(profit), 2)  AS total_profit,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) AS margin_pct,
    COUNT(DISTINCT order_id) AS orders
FROM sales
GROUP BY region
ORDER BY total_profit DESC
""")

,region,total_sales,total_profit,margin_pct,orders
0,West,725457.82,108418.45,14.94,1611
1,East,678499.87,91534.84,13.49,1401
2,South,391721.91,46749.43,11.93,822
3,Central,501239.89,39706.36,7.92,1175


In [19]:
q("""
SELECT
    sub_category,
    ROUND(SUM(profit), 2) AS total_profit
FROM sales
GROUP BY sub_category
HAVING SUM(profit) < 0
ORDER BY total_profit
""")

,sub_category,total_profit
0,Tables,-17725.48
1,Bookcases,-3472.56
2,Supplies,-1189.10


In [20]:
q("""
SELECT
    strftime('%Y', order_date)  AS year,
    ROUND(SUM(sales), 2)        AS total_sales,
    ROUND(SUM(profit), 2)       AS total_profit,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) AS margin_pct
FROM sales
GROUP BY year
ORDER BY year
""")

,year,total_sales,total_profit,margin_pct
0,2014,483966.13,49556.03,10.24
1,2015,470532.51,61618.60,13.10
2,2016,609205.60,81795.17,13.43
3,2017,733215.26,93439.27,12.74


In [21]:
q("""
SELECT
    strftime('%Y-%m', order_date) AS year_month,
    ROUND(SUM(sales), 2)          AS total_sales
FROM sales
GROUP BY year_month
ORDER BY year_month
""")

,year_month,total_sales
0,2014-01,14236.90
1,2014-02,4519.89
2,2014-03,55691.01
3,2014-04,28013.97
4,2014-05,23648.29
5,2014-06,34595.13
6,2014-07,33946.39
7,2014-08,27909.47
8,2014-09,81777.35
9,2014-10,31453.39


In [22]:
q("""
SELECT ROUND(SUM(total_sales), 2) AS check_total
FROM (
    SELECT SUM(sales) AS total_sales
    FROM sales
    GROUP BY strftime('%Y-%m', order_date)
)
""")

,check_total
0,2296919.49


In [23]:
q("""
SELECT
    product_name,
    ROUND(SUM(sales), 2)   AS total_sales,
    ROUND(SUM(profit), 2)  AS total_profit,
    SUM(quantity)          AS units_sold
FROM sales
GROUP BY product_name
ORDER BY total_sales DESC
LIMIT 10
""")

,product_name,total_sales,total_profit,units_sold
0,Canon imageCLASS 2200 Advanced Copier,61599.82,25199.93,20
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,7753.04,31
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08,6
3,HON 5400 Series Task Chairs for Big and Tall,21870.58,0.00,39
4,GBC DocuBind TL300 Electric Binding System,19823.48,2233.51,37
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.50,760.98,48
6,Hewlett Packard LaserJet 3310 Copier,18839.69,6983.88,38
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.90,4094.98,12
8,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17,27
9,High Speed Automatic Electric Letter Opener,17030.31,-262.00,11


In [24]:
q("""
SELECT
    c.customer_name,
    c.segment,
    COUNT(DISTINCT s.order_id) AS orders,
    ROUND(SUM(s.sales), 2)     AS total_sales,
    ROUND(SUM(s.profit), 2)    AS total_profit
FROM sales AS s
INNER JOIN customers AS c
    ON s.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name, c.segment
ORDER BY total_profit DESC
LIMIT 10
""")

,customer_name,segment,orders,total_sales,total_profit
0,Tamara Chand,Corporate,5,19052.22,8981.32
1,Raymond Buch,Consumer,6,15117.34,6976.10
2,Sanjit Chand,Consumer,9,14142.33,5757.41
3,Hunter Lopez,Consumer,6,12873.30,5622.43
4,Adrian Barton,Consumer,10,14473.57,5444.81
5,Tom Ashbrook,Home Office,4,14595.62,4703.79
6,Christopher Martinez,Consumer,4,8954.02,3899.89
7,Keith Dawkins,Corporate,12,8181.26,3038.63
8,Andy Reiter,Consumer,6,6608.45,2884.62
9,Daniel Raglin,Home Office,8,8350.87,2869.08


In [25]:
df_clean.groupby(["Customer ID", "Customer Name", "Segment"]).agg(
    orders=("Order ID", "nunique"),
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum")
).sort_values("total_profit", ascending=False).head(10).round(2)

,,,orders,total_sales,total_profit
Customer ID,Customer Name,Segment,,,
TC-20980,Tamara Chand,Corporate,5,19052.22,8981.32
RB-19360,Raymond Buch,Consumer,6,15117.34,6976.10
SC-20095,Sanjit Chand,Consumer,9,14142.33,5757.41
HL-15040,Hunter Lopez,Consumer,6,12873.30,5622.43
AB-10105,Adrian Barton,Consumer,10,14473.57,5444.81
TA-21385,Tom Ashbrook,Home Office,4,14595.62,4703.79
CM-12385,Christopher Martinez,Consumer,4,8954.02,3899.89
KD-16495,Keith Dawkins,Corporate,12,8181.26,3038.63
AR-10540,Andy Reiter,Consumer,6,6608.45,2884.62


In [26]:
q("""
SELECT ROUND(SUM(s.sales), 2) AS total_sales, ROUND(SUM(s.profit), 2) AS total_profit
FROM sales AS s
INNER JOIN customers AS c ON s.customer_id = c.customer_id
""")

,total_sales,total_profit
0,2296919.49,286409.08


In [27]:
q("""
SELECT
    c.segment,
    ROUND(SUM(s.sales), 2)  AS total_sales,
    ROUND(SUM(s.profit), 2) AS total_profit,
    ROUND(SUM(s.profit) / SUM(s.sales) * 100, 2) AS margin_pct
FROM sales AS s
INNER JOIN customers AS c ON s.customer_id = c.customer_id
GROUP BY c.segment
ORDER BY total_profit DESC
""")

,segment,total_sales,total_profit,margin_pct
0,Consumer,1161401.34,134119.21,11.55
1,Corporate,706146.37,91979.13,13.03
2,Home Office,429371.78,60310.74,14.05


In [28]:
seg = df_clean.groupby("Segment")[["Sales", "Profit"]].sum()
seg["Margin %"] = seg["Profit"] / seg["Sales"] * 100
seg.sort_values("Profit", ascending=False).round(2)

,Sales,Profit,Margin %
Segment,,,
Consumer,1161401.34,134119.21,11.55
Corporate,706146.37,91979.13,13.03
Home Office,429371.78,60310.74,14.05


In [29]:
q("""
SELECT COUNT(DISTINCT order_id) AS orders_2017
FROM sales
WHERE strftime('%Y', order_date) = '2017'
""")

,orders_2017
0,1687


In [30]:
# 1) Each category's share of total sales and total profit
share = df_clean.groupby("Category")[["Sales", "Profit"]].sum()
print((share / share.sum() * 100).round(1))

# 2) Rows with discount >= 30% vs <= 20%
high = df_clean[df_clean["Discount"] >= 0.3]
low = df_clean[df_clean["Discount"] <= 0.2]
print("Rows with discount >= 30%:", len(high),
      "| share of all rows:", round(len(high) / len(df_clean) * 100, 1), "%")
print("Profit at discount >= 30%:", round(high["Profit"].sum(), 2))
print("Profit at discount <= 20%:", round(low["Profit"].sum(), 2))

# 3) Growth from 2014 to 2017
y = df_clean.groupby(df_clean["Order Date"].dt.year)[["Sales", "Profit"]].sum()
print(((y.loc[2017] / y.loc[2014] - 1) * 100).round(1))

                 Sales  Profit
Category                      
Furniture         32.3     6.4
Office Supplies   31.3    42.8
Technology        36.4    50.8
Rows with discount >= 30%: 1392 | share of all rows: 13.9 %
Profit at discount >= 30%: -135364.0
Profit at discount <= 20%: 421773.08
Sales     51.5
Profit    88.6
dtype: float64


In [31]:
q("""
SELECT region,
       ROUND(AVG(discount), 3) AS avg_discount,
       ROUND(100.0 * SUM(CASE WHEN discount >= 0.3 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_rows_discount_30_plus
FROM sales
GROUP BY region
ORDER BY avg_discount DESC
""")

,region,avg_discount,pct_rows_discount_30_plus
0,Central,0.240,27.7
1,South,0.147,10.6
2,East,0.145,16.2
3,West,0.109,3.7


In [32]:
q("""
SELECT sub_category,
       ROUND(AVG(discount), 3) AS avg_discount,
       ROUND(SUM(profit), 2)   AS total_profit
FROM sales
GROUP BY sub_category
ORDER BY avg_discount DESC
""")

,sub_category,avg_discount,total_profit
0,Binders,0.372,30221.76
1,Machines,0.306,3384.76
2,Tables,0.261,-17725.48
3,Bookcases,0.211,-3472.56
4,Chairs,0.170,26602.23
5,Appliances,0.167,18138.01
6,Copiers,0.162,55617.82
7,Phones,0.155,44515.73
8,Furnishings,0.138,13059.14
9,Fasteners,0.082,949.52


In [33]:
q("""
SELECT sub_category, ROUND(SUM(profit), 2) AS profit_central
FROM sales
WHERE region = 'Central'
GROUP BY sub_category
ORDER BY profit_central
""")

,sub_category,profit_central
0,Furnishings,-3906.22
1,Tables,-3559.65
2,Appliances,-2638.62
3,Bookcases,-1997.90
4,Machines,-1486.07
5,Binders,-1043.64
6,Supplies,-661.89
7,Fasteners,236.62
8,Labels,1073.08
9,Art,1195.16


## Validation

I calculated the key numbers independently in Python, SQL and Power BI and compared the results.

### Cross-tool check

| Check | Python | SQL | Power BI |
|---|---|---|---|
| Total Sales | 2,296,919.49 | 2,296,919.49 | 2,296,919.49 |
| Total Profit | 286,409.08 | 286,409.08 | 286,409.08 |
| Total Orders | 5,009 | 5,009 | 5,009 |
| Profit Margin | 12.47% | 12.47% | 12.47% |
| 2017 Sales | 733,215.26 | 733,215.26 | 733.22K |
| 2017 Profit | 93,439.27 | 93,439.27 | 93.44K |
| 2017 Orders | 1,687 | 1,687 | 1,687 |
| Technology Sales | 836,154.03 | 836,154.03 | ~836.15K |
| West Profit | 108,418.45 | 108,418.45 | ~108.42K |

Total Sales, Total Profit, Total Orders and Profit Margin were calculated independently in Python, SQL and Power BI, and all three matched.

### Data-quality log

| Check | Result | Action |
|---|---|---|
| Rows loaded | 9,994 | None |
| Missing values | 0 | None |
| Exact duplicates (ignoring Row ID) | 1 | Removed, leaving 9,993 rows |
| Ship Date before Order Date | 0 | None |
| Invalid Sales / Quantity / Discount | 0 | None |
| Negative-profit rows | 1,870 | Kept on purpose (real losses) |
| Order dates | 2014-01-03 to 2017-12-30 | None |
| Country column | One value only | Left in, not used |